# Mujassam V5 — final test of the V5 model, then the demo (session 1 of 2)

Upload the **V5 adapter zip** chosen on validation (`docs/v5_plan.md`). This notebook:
1. decides the two inference options on the **validation set** (fixed rules, `docs/v5_plan.md` addenda 1 and 3): the feature gate, and **"sampling on"** (if the greedy first try fails: 3 sampled first tries and sampled repair rounds, T=0.7; validation showed greedy repairs repeat the same program). The greedy answers are reused from the candidate's own validation run (same model and settings, saved on Drive); "sampling on" only re-runs the cases greedy failed, because a greedy first-try pass is returned before any sampling;
2. runs **every test set once** with the chosen options: sealed user test u01-u30, held-out part types, reserve m01-m20, in-family tests, original n01-n20 (seen);
3. saves everything and opens the demo app.

Run **`Mujassam_V5_compare_base_V4.ipynb` in a second session at the same time** (base model and V4 on the same new tests).

Time on an A100: **~40 min in the default first-try mode** (`FIRST_TRY_ONLY = True` in cell 0: one attempt per case, the same for every model; validation showed repair attempts never changed a result). Run all; cell 2 asks for the project zip (allow Google Drive), cell 5 for the V5 adapter zip. Every finished case is saved to Drive: after a disconnect, Run all again and it continues.

In [ ]:
# 0. Which epoch of the winner this adapter zip is (Claude tells you after validation): "ep2" or "ep1"
WINNER_EPOCH = "ep2"
FIRST_TRY_ONLY = True   # one attempt per case: fast (validation: repairs never changed a result); False = up to 4
ROUNDS = 1 if FIRST_TRY_ONLY else 4
TAG = "_1try" if FIRST_TRY_ONLY else ""

In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true
import torch
assert torch.cuda.is_available(), "Choose a GPU runtime: Runtime -> Change runtime type -> A100."
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", GPU_NAME, f"({GPU_GB:.0f} GB)")

In [ ]:
# 2. Upload Mujassam_QWEN7B_V5_project.zip
import os, sys, zipfile, shutil, json, hashlib, glob, time, math
from pathlib import Path
from collections import Counter
from google.colab import files

PROJECT = Path("/content/mujassam_mvp_project")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
uploaded = files.upload()
with zipfile.ZipFile(next(iter(uploaded))) as z:
    z.extractall("/content")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("Working directory:", os.getcwd())

# Google Drive keeps the model, the epoch checkpoints and the scores if Colab disconnects.
# Click "Allow" in the pop-up. If you cancel, everything still downloads as before.
DRIVE = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/mujassam_v5")
    DRIVE.mkdir(parents=True, exist_ok=True)
    print("Google Drive mounted: results are also saved to", DRIVE)
except Exception as exc:
    print("Google Drive not mounted (results are only downloaded):", exc)

def to_drive(path):
    """Copy a file to Drive if it is mounted; never stops the run."""
    if DRIVE is None:
        return
    try:
        shutil.copy(path, DRIVE / Path(path).name)
        print("saved to Drive:", DRIVE / Path(path).name)
    except Exception as exc:
        print("Drive copy failed:", exc)

In [ ]:
# 3. Install (CadQuery 2.8 as in every earlier run), then the CPU smoke test of the geometry half
!apt-get -qq install -y libgl1 libxrender1 libglu1-mesa > /dev/null 2>&1
!pip install -q "cadquery==2.8.0" "trimesh>=4.0" 2>&1 | tail -1
!pip install -q "transformers>=4.46" "accelerate>=1.0" "bitsandbytes>=0.44" "peft>=0.13" "trl>=0.12" "datasets>=3.0" "sentence-transformers>=3.0" "chromadb>=0.5" "gradio==6.29.0" 2>&1 | tail -1
!pip uninstall -y -q torchao   # Colab's torchao 0.10 breaks newer peft adapter loading; not used here
!python scripts/smoke_test.py

In [ ]:
# 4. Record the exact library versions of this session
import importlib.metadata as md
VERSIONS = {}
for p in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "datasets", "trl",
          "sentence-transformers", "chromadb", "cadquery", "trimesh", "gradio"]:
    try:
        VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError:
        VERSIONS[p] = None
Path("out").mkdir(exist_ok=True)
Path("out/environment_versions.json").write_text(json.dumps({"gpu": GPU_NAME, **VERSIONS}, indent=2))
!pip freeze > out/environment_freeze.txt
print(json.dumps(VERSIONS, indent=1))

In [ ]:
# 5. Upload the V5 adapter zip (mujassam_v5_<candidate>_adapter.zip) and load it on its own base model
from mujassam.runtime_v3 import load_model, adapter_base
up = files.upload()
shutil.rmtree("/content/winner", ignore_errors=True)
with zipfile.ZipFile(next(iter(up))) as z:
    z.extractall("/content/winner")
found = list(Path("/content/winner").rglob("adapter_config.json"))
assert found, "adapter_config.json is not in that zip - upload the adapter zip, not a results zip"
ADAPTER = str(found[0].parent)
info = json.loads(Path(ADAPTER, "v5_training.json").read_text()) if Path(ADAPTER, "v5_training.json").exists() else {}
WINNER = (info.get("candidate") or {}).get("name", "V5")
BASE_MODEL = adapter_base(ADAPTER)
model, tok = load_model(ADAPTER)
print("Winner:", WINNER, WINNER_EPOCH, "| base:", BASE_MODEL, "| adapter:", ADAPTER)

In [ ]:
# Evaluation helpers: locked sets, Wilson intervals, exact McNemar, one evaluate() for every run
from mujassam.runtime_v3 import build_final_pipeline
from mujassam.evaluate import load_prompts, run_generative_eval, report_view, summarise
from mujassam.monitor import RunLog
from mujassam.partgrid import show_parts

def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

LOCKED = json.loads(Path("data/v5/locked_tests.json").read_text())
SETS = {}
for key, info in LOCKED.items():
    assert sha(info["path"]) == info["sha256"], f"{info['path']} changed since it was locked"
    rows = load_prompts(Path(info["path"]))
    if "ids" in info:
        rows = [r for r in rows if r["id"] in info["ids"]]
    assert len(rows) == info["n"], (key, len(rows))
    SETS[key] = rows
SETS["validation"] = SETS["val_v5"] + SETS["dev_p05_p20"]
print({k: len(v) for k, v in SETS.items()})

def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def mcnemar(a_ok, b_ok):
    """Exact two-sided McNemar p-value for paired pass/fail lists."""
    b = sum(1 for x, y in zip(a_ok, b_ok) if x and not y)
    c = sum(1 for x, y in zip(a_ok, b_ok) if y and not x)
    n = b + c
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(0, min(b, c) + 1)) / 2 ** n)

run_log = RunLog(Path("out/v5_runs.jsonl"), alert_after=10**6)
RESULTS = {}
# Every finished case is saved to Drive at once; after a disconnect, Run all continues with the next case.
RUN_NAME = globals().get("WINNER") or (globals().get("CANDIDATE") or {}).get("name", "run")
CACHE_DIR = (DRIVE / "eval_cache" / RUN_NAME) if globals().get("DRIVE") else None

def evaluate(label, key, pipeline, adapter_off=False):
    tag = f"{label}_{key}".replace(" ", "_")
    cache = (CACHE_DIR / f"{tag}.jsonl") if CACHE_DIR else None
    if adapter_off:
        with model.disable_adapter():
            run = run_generative_eval(pipeline, SETS[key], tag, run_log=run_log, cache_path=cache)
    else:
        run = run_generative_eval(pipeline, SETS[key], tag, run_log=run_log, cache_path=cache)
    rep = report_view(run["records"]) if pipeline.s.n_candidates == 1 else None
    RESULTS[(label, key)] = {"gate": run["records"], "report": rep}
    n = len(run["records"])
    k_gate = sum(r["ok"] for r in run["records"])
    lo, hi = wilson(k_gate, n)
    size = f"size pass {sum(r['ok'] for r in rep)}/{n} | " if rep else ""
    print(f"{label} | {key}: {size}size + requested holes {k_gate}/{n} [{lo:.2f}-{hi:.2f}] | "
          f"first try {sum(r['first_round_ok'] for r in run['records'])}/{n}")
    return run

def split_counts(records, rows):
    by = Counter()
    for rec, row in zip(records, rows):
        s = row.get("split", "dev")
        by[(s, "n")] += 1
        by[(s, "pass")] += int(rec["ok"])
    return {s: f"{by[(s, 'pass')]}/{by[(s, 'n')]}" for s in sorted({k[0] for k in by})}

In [ ]:
# 6. Inference decisions on the VALIDATION set (fixed rules, docs/v5_plan.md addendum 1):
#    gate      - kept if pass_without_missing_holes on dev p05-p20 is higher with the gate than in the report view;
#    sampling  - "sampling on" (3 sampled first tries after a failed greedy try + sampled repairs, T=0.7) is
#                adopted if it passes at least 2 more validation cases (size + requested holes) than greedy.
pipe = build_final_pipeline(model, tok, validator="v3", feature_check="gate", model_id=f"{BASE_MODEL}+mujassam-v5-{WINNER}")

# The candidate notebook already ran exactly this (same adapter, same pipeline, greedy): reuse its saved answers.
if CACHE_DIR:
    src = CACHE_DIR / f"{WINNER}_{WINNER_EPOCH}_validation.jsonl"
    dst = CACHE_DIR / "V5_greedy_validation.jsonl"
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
        print("Reusing the candidate's", WINNER_EPOCH, "validation answers:", src)
evaluate("V5_greedy", "validation", pipe)
recs = RESULTS[("V5_greedy", "validation")]
dev_ids = {r["id"] for r in SETS["dev_p05_p20"]}
dev_gate = [r for r, row in zip(recs["gate"], SETS["validation"]) if row["id"] in dev_ids]
dev_rep = [r for r, row in zip(recs["report"], SETS["validation"]) if row["id"] in dev_ids]
USE_GATE = (summarise(dev_gate).get("pass_without_missing_holes") or 0) > (summarise(dev_rep).get("pass_without_missing_holes") or 0)
greedy_recs = recs["gate"] if USE_GATE else recs["report"]
greedy_ok = sum(r["ok"] for r in greedy_recs)

# "Sampling on" returns the greedy try when it passes on the first try (sampling starts only after a failed
# first try), so those cases are identical; only the other cases are run with sampling.
MODE = "gate" if USE_GATE else "report"
SAMPLING = dict(n_candidates=4, candidate_temperature=0.7, temperature=0.7)
if CACHE_DIR:
    s_cache = CACHE_DIR / f"V5_sampling_{MODE}_validation.jsonl"
    if not s_cache.exists():
        with s_cache.open("w", encoding="utf-8") as fh:
            for r in greedy_recs:
                if r["ok"] and r["rounds"] == 1:
                    fh.write(json.dumps(r, ensure_ascii=False) + "\n")
try:
    assert not FIRST_TRY_ONLY, "first-try mode: sampling (which adds attempts) is not measured"
    pipe_s = build_final_pipeline(model, tok, validator="v3", feature_check=MODE, model_id=pipe.s.model_id, **SAMPLING)
    evaluate(f"V5_sampling_{MODE}", "validation", pipe_s)
    sampling_ok = sum(r["ok"] for r in RESULTS[(f"V5_sampling_{MODE}", "validation")]["gate"])
except Exception as exc:
    print("Sampling could not be measured, keeping greedy:", exc)
    sampling_ok = None
USE_SAMPLING = sampling_ok is not None and sampling_ok >= greedy_ok + 2
DECISIONS = {"winner": WINNER, "epoch": WINNER_EPOCH, "base": BASE_MODEL, "use_gate": USE_GATE,
             "use_sampling": USE_SAMPLING, "sampling": SAMPLING, "validation_greedy": greedy_ok,
             "validation_sampling": sampling_ok, "n_validation": len(SETS["validation"])}
Path("out/v5_decisions.json").write_text(json.dumps(DECISIONS, indent=1))
to_drive("out/v5_decisions.json")
print(json.dumps(DECISIONS, indent=1))
FINAL = build_final_pipeline(model, tok, validator="v3", feature_check=MODE, model_id=pipe.s.model_id,
                             max_rounds=ROUNDS, **(SAMPLING if USE_SAMPLING else {}))

In [ ]:
# 7. Every test set ONCE with the chosen configuration
TESTS = ["user_u01_u30", "heldout_types", "reserve_m01_m20", "in_family_20", "in_family_f01_f12", "original_n01_n20"]
for key in TESTS:
    print(f"\n=== {key} ===")
    evaluate("V5" + TAG, key, FINAL)
    try:
        show_parts({"records": RESULTS[("V5" + TAG, key)]["gate"]}, f"V5 {WINNER} - {key}", f"out/grid_v5_{key}.png", show=False)
    except Exception as exc:
        print("grid picture skipped:", exc)

In [ ]:
# 8. Results table (Wilson 95% intervals)
import pandas as pd
rows = []
for (label, key), res in RESULTS.items():
    n = len(res["gate"])
    k_gate = sum(r["ok"] for r in res["gate"])
    lo, hi = wilson(k_gate, n)
    rows.append({"model": label, "set": key, "n": n,
                 "size pass": f"{sum(r['ok'] for r in res['report'])}/{n}" if res["report"] else "-",
                 "size + requested holes": f"{k_gate}/{n}", "95% CI": f"{lo:.2f}-{hi:.2f}",
                 "first try": f"{sum(r['first_round_ok'] for r in res['gate'])}/{n}"})
table = pd.DataFrame(rows)
display(table)
table.to_csv("out/v5_results_table.csv", index=False)

In [ ]:
# 9. Save everything and download it
bundle = Path("/content/mujassam_v5_final_results.zip")
keep = sorted({f for pattern in ["out/eval_*.json", "out/v5_*", "out/grid_v5_*.png", "out/environment_*"]
               for f in glob.glob(pattern) if Path(f).is_file()})
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in keep:
        z.write(f)
print("Results bundle:", bundle, round(bundle.stat().st_size / 1e6, 2), "MB")
to_drive(bundle)
files.download(str(bundle))

## Demo (V5, with the chosen options)
Uses the V3.1 measurement fixes for the live app. Give the overall size in each request. Download STEP for Onshape, STL for the printer. Generated CAD is not engineering-certified.

In [ ]:
# 10. Launch the app (public link while this session runs)
import app as mujassam_app
DEMO = build_final_pipeline(model, tok, validator="v31", feature_check=MODE, model_id=pipe.s.model_id,
                            **(SAMPLING if USE_SAMPLING else {}))
mujassam_app.PIPELINE = DEMO
mujassam_app.EXAMPLES = [r["prompt"] for r in SETS["dev_p05_p20"][:3]]
mujassam_app.ui().launch(share=True)